In [1]:
import pandas 
import numpy,os
import pydicom,cv2

label=pandas.read_csv("/kaggle/input/rsna-2024-lumbar-spine-degenerative-classification/test_series_descriptions.csv")

print(label)

label_div={"Sagittal T2/STIR" : ["Spinal Canal Stenosis"],
 "Sagittal T1": ["Left Neural Foraminal Narrowing" , "Right Neural Foraminal Narrowing"],
 "Axial T2" : ["Left Subarticular Stenosis" , "Right Subarticular Stenosis"]}

if os.path.exists(f"/kaggle/working/test_wolf")==False:
    os.mkdir(f"/kaggle/working/test_wolf")

for i in zip(numpy.array(label["series_id"]),numpy.array(label["series_description"]),numpy.array(label["study_id"])):
    print(i[0],label_div[i[1]])
    for j in label_div[i[1]]:
        if os.path.exists(f"/kaggle/working/test_wolf/{j}")==False:
            os.mkdir(f"/kaggle/working/test_wolf/{j}")
        print(j)
        for z in os.listdir(f"/kaggle/input/rsna-2024-lumbar-spine-degenerative-classification/test_images/{i[2]}/{i[0]}"):
            cv2.imwrite(f"/kaggle/working/test_wolf/{j}/{z.split('.')[0]}.jpg",cv2.resize(cv2.normalize(pydicom.dcmread(f"/kaggle/input/rsna-2024-lumbar-spine-degenerative-classification/test_images/{i[2]}/{i[0]}/{z}").pixel_array, None, 0, 255, cv2.NORM_MINMAX).astype('uint8'),(256,256)))
            

   study_id   series_id series_description
0  44036939  2828203845        Sagittal T1
1  44036939  3481971518           Axial T2
2  44036939  3844393089   Sagittal T2/STIR
2828203845 ['Left Neural Foraminal Narrowing', 'Right Neural Foraminal Narrowing']
Left Neural Foraminal Narrowing
Right Neural Foraminal Narrowing
3481971518 ['Left Subarticular Stenosis', 'Right Subarticular Stenosis']
Left Subarticular Stenosis
Right Subarticular Stenosis
3844393089 ['Spinal Canal Stenosis']
Spinal Canal Stenosis


In [2]:
import torch
import torch.nn as nn
import torch.nn.parallel
import torch.optim as optim
import torch.nn.functional as F
import torch.utils.data
import torchvision
import torchvision.datasets as dset
import torchvision.transforms as transforms
import torchvision.utils as vutils
import torchvision.transforms.functional as RF
from PIL import Image
import numpy as np
import random,cv2,pandas
import matplotlib.pyplot as plt
import matplotlib.animation as animation

In [3]:
seed = 999
print("Random Seed: ", seed)
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)  # if you are using multi-GPU.
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)

workers = 2
nz = 100
beta1 = 0.5
ngpu=1
ngf,nc = 3,3
ndf = 64

device = torch.device("cuda:0" if (torch.cuda.is_available() and ngpu > 0) else "cpu")

Random Seed:  999


In [4]:
rafire_wolf={"Left Neural Foraminal Narrowing" : int(len(os.listdir(f"/kaggle/working/test_wolf/Left Neural Foraminal Narrowing"))/2), 
             "Right Neural Foraminal Narrowing" : int(len(os.listdir(f"/kaggle/working/test_wolf/Right Subarticular Stenosis"))/2), 
             "Left Subarticular Stenosis" : int(len(os.listdir(f"/kaggle/working/test_wolf/Spinal Canal Stenosis"))/2), 
             "Right Subarticular Stenosis" : int(len(os.listdir(f"/kaggle/working/test_wolf/Right Neural Foraminal Narrowing"))/2),
             "Spinal Canal Stenosis" : int(len(os.listdir(f"/kaggle/working/test_wolf/Left Subarticular Stenosis"))/2)}


for i in ["Left Neural Foraminal Narrowing", "Right Neural Foraminal Narrowing", "Left Subarticular Stenosis",
          "Right Subarticular Stenosis","Spinal Canal Stenosis"]:
    rafire_wolf[i]=RF.to_tensor(cv2.imread(f"/kaggle/working/test_wolf/{i}/{rafire_wolf[i]}.jpg"))
print(rafire_wolf)

{'Left Neural Foraminal Narrowing': tensor([[[0.0118, 0.0314, 0.0118,  ..., 0.0000, 0.0000, 0.0000],
         [0.0118, 0.0353, 0.0275,  ..., 0.0000, 0.0000, 0.0000],
         [0.0235, 0.0392, 0.0353,  ..., 0.0000, 0.0000, 0.0000],
         ...,
         [0.0314, 0.0706, 0.0314,  ..., 0.0000, 0.0000, 0.0000],
         [0.0157, 0.0353, 0.0314,  ..., 0.0000, 0.0000, 0.0000],
         [0.0118, 0.0039, 0.0235,  ..., 0.0000, 0.0000, 0.0000]],

        [[0.0118, 0.0314, 0.0118,  ..., 0.0000, 0.0000, 0.0000],
         [0.0118, 0.0353, 0.0275,  ..., 0.0000, 0.0000, 0.0000],
         [0.0235, 0.0392, 0.0353,  ..., 0.0000, 0.0000, 0.0000],
         ...,
         [0.0314, 0.0706, 0.0314,  ..., 0.0000, 0.0000, 0.0000],
         [0.0157, 0.0353, 0.0314,  ..., 0.0000, 0.0000, 0.0000],
         [0.0118, 0.0039, 0.0235,  ..., 0.0000, 0.0000, 0.0000]],

        [[0.0118, 0.0314, 0.0118,  ..., 0.0000, 0.0000, 0.0000],
         [0.0118, 0.0353, 0.0275,  ..., 0.0000, 0.0000, 0.0000],
         [0.0235, 0.03

In [5]:
def weights_init(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find('BatchNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.devil=nn.Sequential(
            nn.Conv2d(3, 4,3),
            nn.BatchNorm2d(4),
            nn.ReLU(),
            nn.AvgPool2d(2, 2),

            nn.Conv2d(4, 8, 3),
            nn.BatchNorm2d(8),
            nn.AvgPool2d(2, 2),

            nn.Conv2d(8, 16, 3),
            nn.BatchNorm2d(16),
            nn.AvgPool2d(2, 2),

            nn.Conv2d(16, 32, 3),
            nn.BatchNorm2d(32),
            nn.AvgPool2d(2, 2),

            nn.Conv2d(32, 64, 3),
            nn.BatchNorm2d(64),
            nn.AvgPool2d(2, 2),

            nn.Conv2d(64, 128, 3),
            nn.BatchNorm2d(128),
            nn.AvgPool2d(2, 2),

            nn.Flatten(),

            nn.Linear(512, 264),
            nn.ReLU(),
            nn.Linear(264, 132),
            nn.ReLU(),
            nn.Linear(132, 64),
            nn.ReLU(),
            nn.Linear(64, 15),
            nn.Sigmoid()
        )

    def forward(self,x):
        return self.devil(x)

In [6]:
netD = Discriminator().to(device)
if (device.type == 'cuda') and (ngpu > 1):
    netD = nn.DataParallel(netD, list(range(ngpu)))
netD.apply(weights_init)


Discriminator(
  (devil): Sequential(
    (0): Conv2d(3, 4, kernel_size=(3, 3), stride=(1, 1))
    (1): BatchNorm2d(4, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): AvgPool2d(kernel_size=2, stride=2, padding=0)
    (4): Conv2d(4, 8, kernel_size=(3, 3), stride=(1, 1))
    (5): BatchNorm2d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): AvgPool2d(kernel_size=2, stride=2, padding=0)
    (7): Conv2d(8, 16, kernel_size=(3, 3), stride=(1, 1))
    (8): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (9): AvgPool2d(kernel_size=2, stride=2, padding=0)
    (10): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1))
    (11): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (12): AvgPool2d(kernel_size=2, stride=2, padding=0)
    (13): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1))
    (14): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_

In [7]:
import warnings
warnings.filterwarnings('ignore')

In [8]:
rafire_model={"Left Neural Foraminal Narrowing" : "/kaggle/input/rafire_lnfn/pytorch/default/1/2315.pth", 
  "Right Neural Foraminal Narrowing" : "/kaggle/input/rafire_rnfn/pytorch/default/1/3749.pth", 
  "Left Subarticular Stenosis" : "/kaggle/input/rafire_lss/pytorch/default/1/9434.pth",
  "Right Subarticular Stenosis" : "/kaggle/input/rafire_rss/pytorch/default/1/9439.pth",
  "Spinal Canal Stenosis" : "/kaggle/input/rafire_scs/pytorch/default/1/2483.pth"}

transform = transforms.Compose(
    [transforms.Resize(256),transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

rafire_result={}

for i in rafire_model.keys():
    #print(rafire_model[i],rafire_wolf[i])
    dataloader=torch.utils.data.DataLoader(torchvision.datasets.ImageFolder(root=f'/kaggle/input/rsna-rafire/train_wolf/{"_".join(i.split(" ")).lower()}',transform=transform),batch_size=10,shuffle=True,num_workers=2)
    
    netD.load_state_dict(torch.load(rafire_model[i],map_location=torch.device('cpu')))
    model_result=netD(rafire_wolf[i].reshape(1,3,256,256)).detach().view(-1).numpy()
    for j in zip(dataloader.dataset.classes,model_result):
        z_i=j[0].split("_")
        rafire_result[f"44036939_{'_'.join(z_i[:len(z_i)-1])} {z_i[len(z_i)-1]}"]=j[1]
#print(pandas.DataFrame(rafire_result))
for i in rafire_result.keys():
    print(f"{i} : {rafire_result[i]}")


44036939_left_neural_foraminal_narrowing_l1_l2 Moderate : 1.3488300965036615e-06
44036939_left_neural_foraminal_narrowing_l1_l2 Severe : 4.45541439566324e-18
44036939_left_neural_foraminal_narrowing_l1_l2 normal : 0.001363176153972745
44036939_left_neural_foraminal_narrowing_l2_l3 Moderate : 6.85374834574759e-06
44036939_left_neural_foraminal_narrowing_l2_l3 Severe : 5.4254537196424604e-14
44036939_left_neural_foraminal_narrowing_l2_l3 normal : 0.0020256994757801294
44036939_left_neural_foraminal_narrowing_l3_l4 Moderate : 0.02853301167488098
44036939_left_neural_foraminal_narrowing_l3_l4 Severe : 6.44160558227469e-11
44036939_left_neural_foraminal_narrowing_l3_l4 normal : 0.0009966010693460703
44036939_left_neural_foraminal_narrowing_l4_l5 Moderate : 0.02014627307653427
44036939_left_neural_foraminal_narrowing_l4_l5 Severe : 0.0006940937601029873
44036939_left_neural_foraminal_narrowing_l4_l5 normal : 0.0002903933054767549
44036939_left_neural_foraminal_narrowing_l5_s1 Moderate : 0.02

In [9]:
result_temp=pandas.read_csv("/kaggle/input/rsna-2024-lumbar-spine-degenerative-classification/sample_submission.csv")
#print(result_temp)
#zip(result_temp["row_id"],result_temp["normal_mild"],result_temp["moderate"],result_temp["severe"]):
for i in range(len(result_temp["row_id"])):
    
    for j in rafire_result.keys():
        j_z=j.split(" ")
        if j_z[0]==result_temp["row_id"][i]:
            if j_z[1]=="normal":
                result_temp["normal_mild"][i]=rafire_result[j]
            elif j_z[1]=="Moderate": 
                result_temp["moderate"][i]=rafire_result[j]
            elif j_z[1]=="Severe":
                result_temp["severe"][i]=rafire_result[j]
result_temp.to_csv('submission.csv')